In [0]:
import pandas as pd
import numpy as np

In [0]:
%pip install snowflake-connector-python msal --quiet

# Cost per stop

In [0]:
import snowflake.connector

conn = snowflake.connector.connect(
    account       = "MCKESSON-PSAS2",
    user          = "MASOOD.GHASEMI@MCKESSON.CA",
    authenticator = "externalbrowser",
    warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
    role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
    database      = "PRD_PSAS_ANALYTICS_DB",
    schema        = "GOLD_TRANSPORTATION",
)

lastmile_events = pd.read_sql(
    """ SELECT *
FROM VW_TSP_LASTMILE_EVENTS vtle
WHERE 
   DLVRY_ACTL_DATETIME >= TO_TIMESTAMP('2026-05-01 00:00:00');""",
    conn,
)
conn.close()
# lastmile_events.head()
# -- FILL_DC_CD IN ('8170', '8120', '8126')

In [0]:
main_df=lastmile_events.copy()
#Filter 
asn_inc=['FM','RC',"OV","TS"]
main_df=main_df[main_df["ASN_TYPE_CD"].isin(asn_inc)]

# ---- keys ----
# helper column (Excel col P): route | DC | date
#   -> route-level key used to SPREAD the Route Charge (RC) across the route's stops
main_df["helper column"] = (
    main_df["ROUTE_ID"].astype(str)
    + "|"
    + main_df["DC_CD"].astype(str)
    + "|"
    + pd.to_datetime(main_df["DLVRY_ACTL_DATETIME"]).dt.strftime("%Y%m%d")
)
_dlvry_dt = pd.to_datetime(main_df["DLVRY_ACTL_DATETIME"], errors="coerce")
main_df['year']=_dlvry_dt.dt.year
main_df['month']=_dlvry_dt.dt.month


# stop helper (Excel col Q): helper column | stop seq  (stop appended as the LAST component)
#   -> identifies a distinct physical STOP (excludes customer: many customers/totes at one stop = 1 stop)
stop_num = pd.to_numeric(main_df["DELIVERY_SEQ"], errors="coerce")

main_df["stop helper"] = (
    main_df["helper column"]
    + "|"
    + np.where(
        stop_num.fillna(0).eq(0),
        "RETURN",
        stop_num.fillna(0).astype(int).astype(str)
    )
)

# dist stop flag: 1 on the first row of each distinct stop (a stop is counted once regardless of customer)
main_df["dist stop flag"] = (
    ~main_df["stop helper"].duplicated()
).astype(int)

main_df.loc[main_df["stop helper"].eq(""), "dist stop flag"] = 0

# total Route Charge per route
rc_total = (
    main_df.loc[main_df["ASN_TYPE_CD"].eq("RC")]
      .groupby("helper column")["TOTAL_RATE"]
      .sum()
)

# number of DISTINCT FM stops per route
fm_cnt = (
    main_df.loc[
        (main_df["ASN_TYPE_CD"].eq("FM"))
        & (main_df["dist stop flag"].eq(1))
    ]
    .groupby("helper column")
    .size()
)

main_df["_rc_total"] = main_df["helper column"].map(rc_total).fillna(0)
main_df["_fm_cnt"] = main_df["helper column"].map(fm_cnt).fillna(0)

# route has a Route Charge to spread?
_has_rc = main_df["_rc_total"] > 0

main_df["cost per stop"] = np.where(
    main_df["ASN_TYPE_CD"].isin(["RC", "TS"]),
    0,  # RC route charge & TS topside charge are spread over stops, not stops themselves
    np.where(
        ~_has_rc,
        # no RC on this route: FM taken as-is, OV uses the last-mile misc cost
        np.where(
            main_df["ASN_TYPE_CD"].eq("OV"),
            main_df["LASTMILE_TOTAL_COST"],
            main_df["TOTAL_RATE"],
        ),
        # route has RC: spread it evenly across the distinct FM stops
        np.where(
            (main_df["ASN_TYPE_CD"].eq("FM"))
            & (main_df["dist stop flag"].eq(1)),
            main_df["TOTAL_RATE"]
            + np.where(
                main_df["_fm_cnt"] > 0,
                main_df["_rc_total"] / main_df["_fm_cnt"],
                0,
            ),
            np.where(
                (main_df["ASN_TYPE_CD"].eq("FM"))
                & (main_df["dist stop flag"].eq(0)),
                0,
                main_df["TOTAL_RATE"],  # OV kept as-is
            ),
        ),
    ),
)

# ---- Topside (TS): weekly charge spread across distinct stops ----
# TS charge posts once a week (on Monday) in TOPSIDE_COST, keyed by crossdock DC_CD,
# and covers the whole Mon-Sun week. Spread it evenly over that week's DISTINCT stops
# (distinct TRACKING_ID) for the DC_CD, then add the per-stop share to "cost per stop".
_ts_dt = pd.to_datetime(main_df["DLVRY_ACTL_DATETIME"], errors="coerce")
main_df["ts week"] = _ts_dt.dt.to_period("W-SUN").astype(str)          # Mon-Sun week
main_df["ts key"] = main_df["DC_CD"].astype(str) + "|" + main_df["ts week"]

# total topside charge per (DC_CD | week)
ts_total = (
    main_df.loc[main_df["ASN_TYPE_CD"].eq("TS")]
      .groupby("ts key")["TOPSIDE_COST"]
      .sum()
)

# distinct stops (by TRACKING_ID) per (DC_CD | week), delivery stops only
# (FM + OV). RC route charges and TS topside rows are NOT stops, so they're excluded.
ts_stop_cnt = (
    main_df.loc[main_df["ASN_TYPE_CD"].isin(["FM", "OV"])]
      .groupby("ts key")["TRACKING_ID"]
      .nunique()
)

# flag the first row of each distinct stop (by TRACKING_ID) within DC_CD|week, delivery stops only
main_df["ts dist stop flag"] = (
    main_df["ASN_TYPE_CD"].isin(["FM", "OV"])
    & (~main_df.duplicated(subset=["ts key", "TRACKING_ID"]))
).astype(int)

main_df["_ts_total"] = main_df["ts key"].map(ts_total).fillna(0)
main_df["_ts_cnt"] = main_df["ts key"].map(ts_stop_cnt).fillna(0)

# per-stop topside share, added on each distinct stop (0 elsewhere)
_ts_share = np.where(main_df["_ts_cnt"] > 0, main_df["_ts_total"] / main_df["_ts_cnt"], 0)
main_df["cost per stop"] = main_df["cost per stop"] + np.where(
    main_df["ts dist stop flag"].eq(1), _ts_share, 0
)

# TS charge rows are not stops; now that the topside charge has been spread onto the
# FM/OV stops, drop them so they don't distort cost-per-stop counts or averages.
main_df = main_df[main_df["ASN_TYPE_CD"].ne("TS")].copy()

# ---- derived stop counts (no reliable native stop-count field exists) ----
# a delivery stop = a distinct DELIVERY_SEQ (>0) on an FM/OV row; RC route charges and
# the return-to-DC leg (seq 0) are NOT stops.
_seq = pd.to_numeric(main_df["DELIVERY_SEQ"], errors="coerce")
main_df["delivery stop flag"] = (
    main_df["ASN_TYPE_CD"].isin(["FM", "OV"])
    & _seq.gt(0)
    & (~main_df.duplicated(subset=["helper column", "DELIVERY_SEQ"]))
).astype(int)

# distinct stops per route (route | DC | date)
main_df["route_stop_count"] = (
    main_df.groupby("helper column")["delivery stop flag"].transform("sum")
)

# distinct stops per fill DC per month
main_df["fill_dc_stop_count"] = (
    main_df.groupby(["FILL_DC_CD", "year", "month"])["delivery stop flag"].transform("sum")
)

# cleanup helper columns if not needed
main_df.drop(columns=["_rc_total", "_fm_cnt","helper column","stop helper",
                     "dist stop flag","delivery stop flag",
                     "ts week","ts key","ts dist stop flag","_ts_total","_ts_cnt"], inplace=True)

main_df.head()


In [0]:
# tmp=(main_df[main_df['FILL_DC_CD'].isin(['8120'])]
# .groupby(['year','month','FILL_DC_CD','DC_CD','ROUTE_ID'])['cost per stop'].mean().reset_index()
# )
# tmp
# tmp[tmp['ROUTE_ID']=='XD_8120_TopSide']

In [0]:
# main_df=lastmile_events.copy()
# #Filter 
# asn_inc=['FM','RC',"OV"]
# main_df=main_df[main_df["ASN_TYPE_CD"].isin(asn_inc)]

# # ---- keys ----
# # helper column (Excel col P): route | DC | date
# #   -> route-level key used to SPREAD the Route Charge (RC) across the route's stops
# main_df["helper column"] = (
#     main_df["ROUTE_ID"].astype(str)
#     + "|"
#     + main_df["DC_CD"].astype(str)
#     + "|"
#     + pd.to_datetime(main_df["DLVRY_ACTL_DATETIME"]).dt.strftime("%Y%m%d")
# )
# main_df['year']=main_df["DLVRY_ACTL_DATETIME"].dt.year
# main_df['month']=main_df["DLVRY_ACTL_DATETIME"].dt.month


# # stop helper (Excel col Q): helper column | stop seq  (stop appended as the LAST component)
# #   -> identifies a distinct physical STOP (excludes customer: many customers/totes at one stop = 1 stop)
# stop_num = pd.to_numeric(main_df["DELIVERY_SEQ"], errors="coerce")

# main_df["stop helper"] = (
#     main_df["helper column"]
#     + "|"
#     + np.where(
#         stop_num.fillna(0).eq(0),
#         "RETURN",
#         stop_num.fillna(0).astype(int).astype(str)
#     )
# )

# # dist stop flag: 1 on the first row of each distinct stop (a stop is counted once regardless of customer)
# main_df["dist stop flag"] = (
#     ~main_df["stop helper"].duplicated()
# ).astype(int)

# main_df.loc[main_df["stop helper"].eq(""), "dist stop flag"] = 0

# # stops per hour: number of distinct stops on the ROUTE (route-level sum of dist stop flag)
# main_df["stops per hour"] = (
#     main_df.groupby("helper column")["dist stop flag"]
#       .transform("sum")
# )

# # total Route Charge per route
# rc_total = (
#     main_df.loc[main_df["ASN_TYPE_CD"].eq("RC")]
#       .groupby("helper column")["BASE_COST"]
#       .sum()
# )

# # number of DISTINCT FM stops per route
# fm_cnt = (
#     main_df.loc[
#         (main_df["ASN_TYPE_CD"].eq("FM"))
#         & (main_df["dist stop flag"].eq(1))
#     ]
#     .groupby("helper column")
#     .size()
# )

# main_df["_rc_total"] = main_df["helper column"].map(rc_total).fillna(0)
# main_df["_fm_cnt"] = main_df["helper column"].map(fm_cnt).fillna(0)

# # route has a Route Charge to spread?
# _has_rc = main_df["_rc_total"] > 0

# main_df["cost per stop"] = np.where(
#     main_df["ASN_TYPE_CD"].eq("RC"),
#     0,
#     np.where(
#         ~_has_rc,
#         # no RC on this route: FM taken as-is, OV uses the last-mile misc cost
#         np.where(
#             main_df["ASN_TYPE_CD"].eq("OV"),
#             main_df["LASTMILE_MISC_COST"],
#             main_df["BASE_COST"],
#         ),
#         # route has RC: spread it evenly across the distinct FM stops
#         np.where(
#             (main_df["ASN_TYPE_CD"].eq("FM"))
#             & (main_df["dist stop flag"].eq(1)),
#             main_df["BASE_COST"]
#             + np.where(
#                 main_df["_fm_cnt"] > 0,
#                 main_df["_rc_total"] / main_df["_fm_cnt"],
#                 0,
#             ),
#             np.where(
#                 (main_df["ASN_TYPE_CD"].eq("FM"))
#                 & (main_df["dist stop flag"].eq(0)),
#                 0,
#                 main_df["BASE_COST"],  # OV kept as-is
#             ),
#         ),
#     ),
# )

# # cleanup helper columns if not needed
# main_df.drop(columns=["_rc_total", "_fm_cnt","helper column","stop helper",
#                      "dist stop flag","stops per hour"], inplace=True)

# main_df.head()


# Geography multiplier

In [0]:
import snowflake.connector

conn = snowflake.connector.connect(
    account       = "MCKESSON-PSAS2",
    user          = "MASOOD.GHASEMI@MCKESSON.CA",
    authenticator = "externalbrowser",
    warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
    role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
    database      = "PRD_PSAS_ANALYTICS_DB",
    schema        = "GOLD_TRANSPORTATION",
)

cur = conn.cursor()
cur.execute("USE SECONDARY ROLES ALL")

# verify the database is accessible
try:
    cur.execute("USE DATABASE PRD_ENT_PL_DATALAKE_DB")
except Exception as e:
    print(f"Cannot access database: {e}")
    # fallback: list available databases matching the pattern
    cur.execute("SHOW DATABASES LIKE 'PRD_ENT%'")
    print("Available databases:", cur.fetchall())
    conn.close()
    raise

geog_multiplier = pd.read_sql(
    """
    SELECT
    ROUTE_YEAR,
    ROUTE_MONTH,
    ORIGIN,                                 -- = xdock_code, e.g. XD_8170_AZ_Yuma.AVBV
    TOTAL_LEG_MILES,
    WEIGHTED_MILES_PER_STOP,
    MEDIAN_ROUTE_DAY_MILES_PER_STOP,
    P75_MILES_PER_STOP,
    MAX_MILES_PER_STOP,
    ROUND(
        LEAST(1.50,
              POWER(MEDIAN_ROUTE_DAY_MILES_PER_STOP / NULLIF(BASELINE_MEDIAN, 0), 0.30)),
        2
    ) AS GEOGRAPHY_MULTIPLIER,
    ROUTE_DAYS,
    TOTAL_STOPS
FROM (
    SELECT
        os.*,
        -- baseline = min median among non-crossdock origins, WITHIN each year+month
        MIN(CASE WHEN ORIGIN NOT LIKE 'XD_%'
                 THEN MEDIAN_ROUTE_DAY_MILES_PER_STOP END)
            OVER (PARTITION BY ROUTE_YEAR, ROUTE_MONTH) AS BASELINE_MEDIAN
    FROM (
        -- origin_summary (now per ORIGIN + year + month)
        SELECT
            YEAR(ROUTE_DATE)                                               AS ROUTE_YEAR,
            MONTH(ROUTE_DATE)                                              AS ROUTE_MONTH,
            ORIGIN,
            COUNT(*)                                                        AS ROUTE_DAYS,
            SUM(DISTINCT_STOPS)                                             AS TOTAL_STOPS,
            ROUND(SUM(TOTAL_LEG_MILES), 2)                                  AS TOTAL_LEG_MILES,
            ROUND(SUM(TOTAL_LEG_MILES) / NULLIF(SUM(DISTINCT_STOPS), 0), 2) AS WEIGHTED_MILES_PER_STOP,
            ROUND(MEDIAN(MILES_PER_STOP), 2)                               AS MEDIAN_ROUTE_DAY_MILES_PER_STOP,
            ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY MILES_PER_STOP), 2) AS P75_MILES_PER_STOP,
            ROUND(MAX(MILES_PER_STOP), 2)                                  AS MAX_MILES_PER_STOP
        FROM (
            -- route_day_summary
            SELECT
                ORIGIN,
                FILL_DC,
                DELIVERY_ROUTE_ID,
                ROUTE_DATE,
                COUNT(DISTINCT DELIVERY_STOP_ID)                             AS DISTINCT_STOPS,
                SUM(LEG_MILES)                                               AS TOTAL_LEG_MILES,
                SUM(LEG_MILES) / NULLIF(COUNT(DISTINCT DELIVERY_STOP_ID), 0) AS MILES_PER_STOP
            FROM (
                -- leg_miles
                SELECT
                    DELIVERY_ROUTE_ID,
                    ORIGIN,
                    FILL_DC,
                    ROUTE_DATE,
                    DELIVERY_STOP_ID,
                    3959 * ACOS(
                        LEAST(1, GREATEST(-1,
                            COS(RADIANS(PREV_STOP_LAT)) * COS(RADIANS(DESTINATION_LATITUDE))
                            * COS(RADIANS(DESTINATION_LONGITUDE) - RADIANS(PREV_STOP_LON))
                            + SIN(RADIANS(PREV_STOP_LAT)) * SIN(RADIANS(DESTINATION_LATITUDE))
                        ))
                    ) AS LEG_MILES
                FROM (
                    -- sequenced
                    SELECT
                        DELIVERY_ROUTE_ID,
                        DELIVERY_STOP_ID,
                        ORIGIN,
                        FILL_DC,
                        ROUTE_DATE,
                        DESTINATION_LATITUDE,
                        DESTINATION_LONGITUDE,
                        COALESCE(
                            LAG(DESTINATION_LATITUDE) OVER (
                                PARTITION BY DELIVERY_ROUTE_ID, ORIGIN, FILL_DC, ROUTE_DATE
                                ORDER BY TRY_TO_NUMBER(DELIVERY_STOP_ID), DELIVERY_STOP_ID
                            ),
                            ORIGIN_LATITUDE
                        ) AS PREV_STOP_LAT,
                        COALESCE(
                            LAG(DESTINATION_LONGITUDE) OVER (
                                PARTITION BY DELIVERY_ROUTE_ID, ORIGIN, FILL_DC, ROUTE_DATE
                                ORDER BY TRY_TO_NUMBER(DELIVERY_STOP_ID), DELIVERY_STOP_ID
                            ),
                            ORIGIN_LONGITUDE
                        ) AS PREV_STOP_LON
                    FROM (
                        -- route_stops  (ORIGIN redefined as the xdock_code)
                        SELECT DISTINCT
                            DELIVERY_ROUTE_ID,
                            DELIVERY_STOP_ID,
                            ORIGIN || '.' || COURIER_SCAC_CODE AS ORIGIN,
                            FILL_DC,
                            CAST(CONTRACTUAL_DELIVERY_DATE_AND_TIME AS DATE) AS ROUTE_DATE,
                            DESTINATION_LATITUDE,
                            DESTINATION_LONGITUDE,
                            ORIGIN_LATITUDE,
                            ORIGIN_LONGITUDE
                        FROM PRD_ENT_PL_DATALAKE_DB.TRANSVOYANT.V_ASN_SHIPMENT
                        WHERE DELIVERY_ROUTE_ID IS NOT NULL
                          AND DELIVERY_STOP_ID IS NOT NULL
                          AND DESTINATION_LATITUDE IS NOT NULL
                          AND DESTINATION_LONGITUDE IS NOT NULL
                          AND COURIER_SCAC_CODE IS NOT NULL
                        --  AND FILL_DC = '8170'
                          AND CONTRACTUAL_DELIVERY_DATE_AND_TIME >= TO_TIMESTAMP('2026-05-01 00:00:00')
                    ) route_stops
                ) sequenced
                WHERE PREV_STOP_LAT IS NOT NULL
                  AND PREV_STOP_LON IS NOT NULL
            ) leg_miles
            GROUP BY ORIGIN, FILL_DC, DELIVERY_ROUTE_ID, ROUTE_DATE
        ) route_day_summary
        WHERE DISTINCT_STOPS > 1
        GROUP BY
            YEAR(ROUTE_DATE),
            MONTH(ROUTE_DATE),
            ORIGIN
    ) os
) with_baseline
ORDER BY ROUTE_YEAR, ROUTE_MONTH, MEDIAN_ROUTE_DAY_MILES_PER_STOP DESC;
    """,
    conn,
)
conn.close()


In [0]:
# import snowflake.connector

# conn = snowflake.connector.connect(
#     account       = "MCKESSON-PSAS2",
#     user          = "MASOOD.GHASEMI@MCKESSON.CA",
#     authenticator = "externalbrowser",
#     warehouse     = "PRD_PSAS_ANALYTICS_TRANSPORTATION_WH",
#     role          = "PRD_PSAS_ANALYTICS_TRANSPORTATION_FR",
#     database      = "PRD_PSAS_ANALYTICS_DB",
#     schema        = "GOLD_TRANSPORTATION",
# )

# geog_multiplier = pd.read_sql(
#     """
#     SELECT
#     ROUTE_YEAR,
#     ROUTE_MONTH,
#     ORIGIN,                                 -- = xdock_code, e.g. XD_8170_AZ_Yuma.AVBV
#     TOTAL_LEG_MILES,
#     WEIGHTED_MILES_PER_STOP,
#     MEDIAN_ROUTE_DAY_MILES_PER_STOP,
#     P75_MILES_PER_STOP,
#     MAX_MILES_PER_STOP,
#     ROUND(
#         LEAST(1.50,
#               POWER(MEDIAN_ROUTE_DAY_MILES_PER_STOP / NULLIF(BASELINE_MEDIAN, 0), 0.30)),
#         2
#     ) AS GEOGRAPHY_MULTIPLIER,
#     ROUTE_DAYS,
#     TOTAL_STOPS
# FROM (
#     SELECT
#         os.*,
#         -- baseline = min median among non-crossdock origins, WITHIN each year+month
#         MIN(CASE WHEN ORIGIN NOT LIKE 'XD_%'
#                  THEN MEDIAN_ROUTE_DAY_MILES_PER_STOP END)
#             OVER (PARTITION BY ROUTE_YEAR, ROUTE_MONTH) AS BASELINE_MEDIAN
#     FROM (
#         -- origin_summary (now per ORIGIN + year + month)
#         SELECT
#             YEAR(ROUTE_DATE)                                               AS ROUTE_YEAR,
#             MONTH(ROUTE_DATE)                                              AS ROUTE_MONTH,
#             ORIGIN,
#             COUNT(*)                                                        AS ROUTE_DAYS,
#             SUM(DISTINCT_STOPS)                                             AS TOTAL_STOPS,
#             ROUND(SUM(TOTAL_LEG_MILES), 2)                                  AS TOTAL_LEG_MILES,
#             ROUND(SUM(TOTAL_LEG_MILES) / NULLIF(SUM(DISTINCT_STOPS), 0), 2) AS WEIGHTED_MILES_PER_STOP,
#             ROUND(MEDIAN(MILES_PER_STOP), 2)                               AS MEDIAN_ROUTE_DAY_MILES_PER_STOP,
#             ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY MILES_PER_STOP), 2) AS P75_MILES_PER_STOP,
#             ROUND(MAX(MILES_PER_STOP), 2)                                  AS MAX_MILES_PER_STOP
#         FROM (
#             -- route_day_summary
#             SELECT
#                 ORIGIN,
#                 FILL_DC,
#                 DELIVERY_ROUTE_ID,
#                 ROUTE_DATE,
#                 COUNT(DISTINCT DELIVERY_STOP_ID)                             AS DISTINCT_STOPS,
#                 SUM(LEG_MILES)                                               AS TOTAL_LEG_MILES,
#                 SUM(LEG_MILES) / NULLIF(COUNT(DISTINCT DELIVERY_STOP_ID), 0) AS MILES_PER_STOP
#             FROM (
#                 -- leg_miles
#                 SELECT
#                     DELIVERY_ROUTE_ID,
#                     ORIGIN,
#                     FILL_DC,
#                     ROUTE_DATE,
#                     DELIVERY_STOP_ID,
#                     3959 * ACOS(
#                         LEAST(1, GREATEST(-1,
#                             COS(RADIANS(PREV_STOP_LAT)) * COS(RADIANS(DESTINATION_LATITUDE))
#                             * COS(RADIANS(DESTINATION_LONGITUDE) - RADIANS(PREV_STOP_LON))
#                             + SIN(RADIANS(PREV_STOP_LAT)) * SIN(RADIANS(DESTINATION_LATITUDE))
#                         ))
#                     ) AS LEG_MILES
#                 FROM (
#                     -- sequenced
#                     SELECT
#                         DELIVERY_ROUTE_ID,
#                         DELIVERY_STOP_ID,
#                         ORIGIN,
#                         FILL_DC,
#                         ROUTE_DATE,
#                         DESTINATION_LATITUDE,
#                         DESTINATION_LONGITUDE,
#                         COALESCE(
#                             LAG(DESTINATION_LATITUDE) OVER (
#                                 PARTITION BY DELIVERY_ROUTE_ID, ORIGIN, FILL_DC, ROUTE_DATE
#                                 ORDER BY TRY_TO_NUMBER(DELIVERY_STOP_ID), DELIVERY_STOP_ID
#                             ),
#                             ORIGIN_LATITUDE
#                         ) AS PREV_STOP_LAT,
#                         COALESCE(
#                             LAG(DESTINATION_LONGITUDE) OVER (
#                                 PARTITION BY DELIVERY_ROUTE_ID, ORIGIN, FILL_DC, ROUTE_DATE
#                                 ORDER BY TRY_TO_NUMBER(DELIVERY_STOP_ID), DELIVERY_STOP_ID
#                             ),
#                             ORIGIN_LONGITUDE
#                         ) AS PREV_STOP_LON
#                     FROM (
#                         -- route_stops  (ORIGIN redefined as the xdock_code)
#                         SELECT DISTINCT
#                             DELIVERY_ROUTE_ID,
#                             DELIVERY_STOP_ID,
#                             ORIGIN || '.' || COURIER_SCAC_CODE AS ORIGIN,
#                             FILL_DC,
#                             CAST(CONTRACTUAL_DELIVERY_DATE_AND_TIME AS DATE) AS ROUTE_DATE,
#                             DESTINATION_LATITUDE,
#                             DESTINATION_LONGITUDE,
#                             ORIGIN_LATITUDE,
#                             ORIGIN_LONGITUDE
#                         FROM PRD_ENT_PL_DATALAKE_DB.TRANSVOYANT.V_ASN_SHIPMENT
#                         WHERE DELIVERY_ROUTE_ID IS NOT NULL
#                           AND DELIVERY_STOP_ID IS NOT NULL
#                           AND DESTINATION_LATITUDE IS NOT NULL
#                           AND DESTINATION_LONGITUDE IS NOT NULL
#                           AND COURIER_SCAC_CODE IS NOT NULL
#                         --  AND FILL_DC = '8170'
#                           AND CONTRACTUAL_DELIVERY_DATE_AND_TIME >= TO_TIMESTAMP('2026-05-01 00:00:00')
#                     ) route_stops
#                 ) sequenced
#                 WHERE PREV_STOP_LAT IS NOT NULL
#                   AND PREV_STOP_LON IS NOT NULL
#             ) leg_miles
#             GROUP BY ORIGIN, FILL_DC, DELIVERY_ROUTE_ID, ROUTE_DATE
#         ) route_day_summary
#         WHERE DISTINCT_STOPS > 1
#         GROUP BY
#             YEAR(ROUTE_DATE),
#             MONTH(ROUTE_DATE),
#             ORIGIN
#     ) os
# ) with_baseline
# ORDER BY ROUTE_YEAR, ROUTE_MONTH, MEDIAN_ROUTE_DAY_MILES_PER_STOP DESC;
#     """,
#     conn,
# )
# conn.close()


In [0]:
geog_mult=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/geog_multiplier_alldocks.csv')

In [0]:
geog_mult.head(3)

## add to main df

In [0]:
main_df['DC_CD'] = main_df['DC_CD'].astype(str).str.upper().str.strip()
geog_mult['XDOCK'] = geog_mult['ORIGIN'].astype(str).str.upper().str.strip()

main_geog_mult=(main_df
 .merge(geog_mult[['XDOCK','GEOGRAPHY_MULTIPLIER','ROUTE_YEAR','ROUTE_MONTH']]
        ,left_on=['DC_CD','year','month']
        ,right_on=['XDOCK','ROUTE_YEAR','ROUTE_MONTH'])
)
main_geog_mult.head(3)

In [0]:
len(set(main_geog_mult['FILL_DC_CD']))

# Shipment multiplier

In [0]:
# file=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/McKesson Outbound Data 5.1 - 6.30.xlsx', header=0)

# health check file for May 2026
# file=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/8170_May 2026_HC.xlsx', header=0)

# file_df=file.copy()

# asn=['FM', 'OV', 'RC']
# file_df=file_df[file_df['ASN Type'].isin(asn)]

In [0]:
# file_df['ASN Type'].unique()

In [0]:
# asnshipment=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/V_ASN_SHIPMENT_202607151058.csv', header=0)
# asnshipment=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/V_ASN_SHIPMENT_202607151232.csv', header=0)
# asnshipment_df=asnshipment.copy()

# SELECT
#     ORIGIN || '.' || COURIER_SCAC_CODE AS XDock,
#     DATE(CONTRACTUAL_DELIVERY_DATE_AND_TIME) AS date,
#     year(CONTRACTUAL_DELIVERY_DATE_AND_TIME) AS year,
#     month(CONTRACTUAL_DELIVERY_DATE_AND_TIME) AS month,
#     COUNT(DVS_RECORD_KEY) AS DVS_RECORD_KEY_COUNT
# FROM PRD_ENT_DL_US_PSAS_MISC_DB.TRANSVOYANT.V_ASN_SHIPMENT
# WHERE CONTRACTUAL_DELIVERY_DATE_AND_TIME >= TO_TIMESTAMP('2026-05-01 00:00:00')
#   AND FILL_DC IN ('8170', '8120', '8126')
# GROUP BY
#     ORIGIN || '.' || COURIER_SCAC_CODE,
#     CONTRACTUAL_DELIVERY_DATE_AND_TIME,
# year(CONTRACTUAL_DELIVERY_DATE_AND_TIME),
# month(CONTRACTUAL_DELIVERY_DATE_AND_TIME);

# asnshipment_df_gp=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/asnshipments_gp_ym.csv', header=0)

#remove filter on DC before running above qry
asnshipment_df_gp=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/asnshipment_df_gp.csv', header=0)

In [0]:
# asnshipment_df=asnshipment_df[asnshipment_df['ORIGIN']!='8170'].copy()
# asnshipment_df['XDock']=asnshipment_df['ORIGIN']+"."+asnshipment_df['COURIER_SCAC_CODE']

In [0]:
# asnshipment_df_gp.columns

In [0]:
# asnshipment_df['date'] = (
#     pd.to_datetime(
#         asnshipment_df['CONTRACTUAL_DELIVERY_DATE_AND_TIME'],
#         errors='coerce'
#     )
#     .dt.date
# )

# gp_cols = ['XDock', 'date']

# # Total shipments per day per XDock
# asnshipment_df_gp = (
#     asnshipment_df
#     .groupby(gp_cols, dropna=False)['DVS_RECORD_KEY']
#     .count()
#     .reset_index(name='DVS_RECORD_KEY_COUNT')
# )

# Average daily shipments per XDock
avg_daily_shipments = (
    asnshipment_df_gp
    .groupby(['XDOCK','YEAR', 'MONTH'])['DVS_RECORD_KEY_COUNT']
    .mean()
    .reset_index(name='daily_shipment_cnt')
)

# Baseline across all XDocks, year month
baseline_ship = avg_daily_shipments['daily_shipment_cnt'].median()

print("baseline_ship:", baseline_ship)

# Multiplier
avg_daily_shipments['shipment_norm_multiplier'] = (
     avg_daily_shipments['daily_shipment_cnt'] / baseline_ship
)

# Quantile clipping based on multiplier distribution
lower = avg_daily_shipments['shipment_norm_multiplier'].quantile(0.1)
upper = avg_daily_shipments['shipment_norm_multiplier'].quantile(0.9)

avg_daily_shipments['shipment_norm_multiplier_qt'] = (
    avg_daily_shipments['shipment_norm_multiplier']
    .clip(lower=lower, upper=upper)
)

xdock_shipment_multiplier = avg_daily_shipments.copy()

display(xdock_shipment_multiplier)


In [0]:
# cd_avg_ship_cnt.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/temp/cd_avg_ship_cnt.csv')

# xdock_shipment_multiplier.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/xdock_shipment_multiplier.csv')

# file_shipment.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/file_shipment.csv')

# file_df.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/temp/file_df.csv')

# asnshipment_df.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/temp/asnshipment_df.csv')

## add to main df

In [0]:
main_df['DC_CD'] = main_df['DC_CD'].astype(str).str.upper().str.strip()
xdock_shipment_multiplier['XDOCK'] = xdock_shipment_multiplier['XDOCK'].astype(str).str.upper().str.strip()

main_ship_geog_mult=(main_geog_mult
 .merge(xdock_shipment_multiplier[['XDOCK','shipment_norm_multiplier_qt','YEAR', 'MONTH']]
        ,left_on=['DC_CD','year', 'month']
        ,right_on=['XDOCK','YEAR', 'MONTH'])
)

In [0]:
len(set(main_ship_geog_mult['FILL_DC_CD']))

# Delivery Density

In [0]:
# transvoyant=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/TransVoyant_May_8170.xlsx', header=0)

# trans1=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/transvoyant_73026_1.csv', header=0)
# trans2=pd.read_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/transvoyant_73026_2.csv', header=0)
# transvoyant_df=pd.concat([trans1,trans2], ignore_index=True)

transvoyant_df_0=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/TransVoyant_May-July_8120, 8126, 8170.xlsx', header=0)

transvoyant_df_1=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/tv_combined_1.xlsx', header=0)

transvoyant_df_2=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/tv_combined_2.xlsx', header=0)


In [0]:
transvoyant_df = pd.concat([transvoyant_df_0,transvoyant_df_1, transvoyant_df_2], axis=0, ignore_index=True)

In [0]:
# add Xdock code
name_to_scac = (
    main_df[["CARRIER_SCAC_NAME", "CARRIER_SCAC_CD"]]
        .dropna()
        .drop_duplicates()
        .set_index("CARRIER_SCAC_NAME")["CARRIER_SCAC_CD"]
        .to_dict()
)

# map courier -> SCAC on the SAME df you assign into
_scac = transvoyant_df["Courier Name"].map(name_to_scac)

transvoyant_df["XDOCK"] = np.where(
    transvoyant_df["X Dock"].notna() & _scac.notna(),
    transvoyant_df["X Dock"].astype(str) + "." + _scac.astype(str),
    np.nan,
)

# Outlier removal from total distance 
col = 'Total distance traveled, mi'

transvoyant_df[col] = pd.to_numeric(
    transvoyant_df[col],
    errors='coerce'
)

q1 = transvoyant_df[col].quantile(0.25)
q3 = transvoyant_df[col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (
    (transvoyant_df[col] < lower_bound) |
    (transvoyant_df[col] > upper_bound)
)

outlier_count = outlier_mask.sum()
total_count = len(transvoyant_df)
outlier_pct = 100 * outlier_count / total_count

print(f'Outlier count: {outlier_count:,}')
print(f'Total count: {total_count:,}')
print(f'Outlier %: {outlier_pct:.2f}%')

# removes only rows with outlier distance values
transvoyant_df = transvoyant_df.loc[~outlier_mask].copy()

print(f'Rows remaining: {len(transvoyant_df):,}')
print("Min,Max:",transvoyant_df[col].min(),transvoyant_df[col].max())


In [0]:
# Extract day from Delivery ETA Start
# transvoyant_df['Delivery_Date'] = (
#     pd.to_datetime(
#         transvoyant_df['Delivery ETA Start']
#             .astype(str)
#             .str.replace(r'\s+(MST|MDT|CST|CDT|PST|PDT)$', '', regex=True),
#         errors='coerce'
#     )
#     .dt.date
# )
# Extract day / year / month from Delivery ETA Start

_delivery_dt = pd.to_datetime(
    transvoyant_df['Delivery ETA Start']
        .astype(str)
        .str.replace(r'\s+[A-Z]{2,4}$', '', regex=True),
    errors='coerce'
)

transvoyant_df['Delivery_Date'] = _delivery_dt.dt.date
transvoyant_df['year'] = _delivery_dt.dt.year
transvoyant_df['month'] = _delivery_dt.dt.month

# Max distance and stop count per Route / Day

transvoyant_max_dist = (
    transvoyant_df.groupby(
        [
            'Fill DC',
            'XDOCK',
            'Delivery Route ID',
            'Delivery_Date',
            'year',
            'month'
        ],
        as_index=False
    )
    .agg(
        total_miles=('Total distance traveled, mi', 'max'),
        stop_count=('Delivery Stop ID', 'nunique')
    )
)

# Roll up to XDock / Day

total_xd_daily_miles = (
    transvoyant_max_dist
    .groupby(
        [
            'Fill DC',
            'XDOCK',
            'Delivery_Date',
            'year',
            'month'
        ],
        as_index=False
    )
    .agg(
        total_miles=('total_miles', 'sum'),
        stop_count=('stop_count', 'sum')
    )
    .assign(
        miles_per_stop=lambda df: df['total_miles'] / df['stop_count']
    )
)

# Average miles per stop by XDock / Year / Month

xd_avg_daily_miles_per_stop = (
    total_xd_daily_miles
    .groupby(
        ['XDOCK', 'year', 'month'],
        as_index=False
    )
    .agg(
        avg_miles_per_stop=('miles_per_stop', 'mean')
    )
)

# Monthly baseline (median across XDocks)

xd_avg_daily_miles_per_stop['baseline_miles_per_stop'] = (
    xd_avg_daily_miles_per_stop
    .groupby(['year', 'month'])['avg_miles_per_stop']
    .transform('median')
)

print(
    xd_avg_daily_miles_per_stop[
        ['year', 'month', 'baseline_miles_per_stop']
    ].drop_duplicates()
)

# Multiplier

xd_avg_daily_miles_per_stop['miles_per_stop_norm_multiplier'] = (
    xd_avg_daily_miles_per_stop['avg_miles_per_stop']
    / xd_avg_daily_miles_per_stop['baseline_miles_per_stop']
)

# Quantile clipping within month

lower = (
    xd_avg_daily_miles_per_stop
    .groupby(['year', 'month'])['miles_per_stop_norm_multiplier']
    .transform(lambda x: x.quantile(0.10))
)

upper = (
    xd_avg_daily_miles_per_stop
    .groupby(['year', 'month'])['miles_per_stop_norm_multiplier']
    .transform(lambda x: x.quantile(0.90))
)

xd_avg_daily_miles_per_stop['miles_per_stop_norm_multiplier_qt'] = (
    xd_avg_daily_miles_per_stop['miles_per_stop_norm_multiplier']
    .clip(lower=lower, upper=upper)
)

display(xd_avg_daily_miles_per_stop)


# display(xd_avg_daily_miles_per_stop)
# total_xd_daily_miles['Fill DC']=total_xd_daily_miles['Fill DC'].astype("str")

## add to main df

In [0]:
xd_avg_daily_miles_per_stop['XDOCK'] = xd_avg_daily_miles_per_stop['XDOCK'].astype(str).str.upper().str.strip()

main_geog_dense_ship_mult=(main_ship_geog_mult
 .merge(xd_avg_daily_miles_per_stop[['XDOCK','miles_per_stop_norm_multiplier_qt','year','month']]
        ,left_on=['DC_CD','year','month']
        ,right_on=['XDOCK','year','month'])
)

In [0]:
len(set(main_geog_dense_ship_mult['FILL_DC_CD']))

In [0]:
# xd_avg_daily_miles_per_stop.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/xd_avg_daily_miles_per_stop.csv')

# Calculate cost per mile and normalize

In [0]:
main_geog_dense_ship_mult.head(3)

In [0]:
# Geometric mean
main_geog_dense_ship_mult['geo_mean'] = (main_geog_dense_ship_mult['shipment_norm_multiplier_qt'] * main_geog_dense_ship_mult['miles_per_stop_norm_multiplier_qt']*main_geog_dense_ship_mult['miles_per_stop_norm_multiplier_qt'])** (1/3)

main_geog_dense_ship_mult['normalized_cost']=main_geog_dense_ship_mult['cost per stop']/main_geog_dense_ship_mult['geo_mean']

#normalized geography
main_geog_dense_ship_mult['normalized_cost_geography']=(main_geog_dense_ship_mult['cost per stop']/
main_geog_dense_ship_mult['GEOGRAPHY_MULTIPLIER'])

#normalized shipments
main_geog_dense_ship_mult['normalized_cost_shipment']=(main_geog_dense_ship_mult['cost per stop']/
main_geog_dense_ship_mult['shipment_norm_multiplier_qt'])

#normalized density
main_geog_dense_ship_mult['normalized_cost_density']=(main_geog_dense_ship_mult['cost per stop']/
main_geog_dense_ship_mult['miles_per_stop_norm_multiplier_qt'])

print(main_geog_dense_ship_mult.groupby(['FILL_DC_CD','YEAR','MONTH'])['DC_CD'].nunique())
main_geog_dense_ship_mult.head()


In [0]:
lm_agg=main_geog_dense_ship_mult.groupby(['FILL_DC_CD','YEAR','MONTH','DC_CD'])['normalized_cost'].mean().reset_index()
lm_agg


In [0]:
lm_agg.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/lm_agg.csv')

In [0]:
# main_geog_dense_ship_mult.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/lm_geog_dense_ship_mult.csv')
main_geog_dense_ship_mult.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/main_df_lm.csv')

## Merge LM Cost with CleanSheet 

In [0]:
# cleansheet_may=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/cleansheet_may.xlsx')
cleansheet_may=pd.read_excel('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/cleansheet_may_v5.xlsx')

In [0]:
LM_CS_df=(main_geog_dense_ship_mult.merge(cleansheet_may,
                                 left_on=['DC_CD','year','month'],
                                 right_on=['Distribution Center','year','month']
                                 ,how='left'))
# LM_CS_df

In [0]:
LM_CS_df[LM_CS_df['FILL_DC_CD'].isin(['8170', '8120', '8126'])].to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/LM_CS_8170&8120&8126_0814.csv')

In [0]:
LM_CS_df.to_csv('/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/data/output/LM_CS_0812.csv')

In [0]:
(LM_CS_df[LM_CS_df['FILL_DC_CD'].isin(['8120'])]
.groupby(['year','month','FILL_DC_CD','DC_CD'])['cost per stop'].mean().reset_index()
)

In [0]:
LM_CS_df[LM_CS_df['ROUTE_ID']=='XD_8120_TopSide'].groupby(['year','month'])['cost per stop'].mean().reset_index()

In [0]:
LM_CS_df.columns

# For databricks app

In [0]:
import os

dashboard_cols = [
    "FILL_DC_CD",
    "XDOCK",
    "YEAR",
    "MONTH",
    "cost per stop",
    "normalized_cost",
    "Cleansheet Cost Per Stop Conservative",
    "Cleansheet Cost Per Stop Aggressive",
    # normalization multipliers (drive the correlation / drivers tab)
    "GEOGRAPHY_MULTIPLIER",
    "miles_per_stop_norm_multiplier_qt",
    "shipment_norm_multiplier_qt",
    "geo_mean",
]


present = [c for c in dashboard_cols if c in LM_CS_df.columns]
missing = [c for c in dashboard_cols if c not in LM_CS_df.columns]

# --> app folder's data/ subfolder
# OUT = "/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/apps/lm_cost/data/LM_CS_slim.csv"
OUT = "/Workspace/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/apps/lm_cost/data/LM_CS_slim.csv.gz"
LM_CS_df[present].to_csv(OUT, index=False, compression="gzip")

# OUT = "'/Users/sgt4gul@mckessoncorp.onmicrosoft.com/Last Mile/analysis/apps/lm_cost/data/LM_CS_slim.csv"
# os.makedirs(os.path.dirname(OUT), exist_ok=True)

# LM_CS_df[present].to_csv(OUT, index=False) 

# print(f"Wrote {OUT}: {len(present)} cols | missing: {missing}")
print("wrote:", OUT, "| size MB:", round(os.path.getsize(OUT) / 1e6, 2))


In [0]:
# ============================================================
# LAST MILE COST OUTLIER ANALYSIS
# Notebook version, no Streamlit
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from IPython.display import display

import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [0]:
# ============================================================
# CONFIG
# ============================================================

TARGET = "cost per stop"

GROUP_COLS = [
    "DC_CD",
    "year",
    "month",
    "CARRIER_SCAC_CD"
]

RANDOM_STATE = 42

MAX_MODEL_ROWS = 90000

MIN_GROUP_N = 30

OVERPAY_STRONG = 0.20
OVERPAY_POSSIBLE = 0.10
UNDERPAY_STRONG = -0.20
UNDERPAY_POSSIBLE = -0.10

EXPORT_EXCEL = True
EXCEL_OUTPUT_PATH = "lastmile_outlier_scorecard.xlsx"


# ============================================================
# HELPERS
# ============================================================

def existing_cols(df, columns):
    return [c for c in columns if c in df.columns]


def safe_div(a, b):
    return np.where((pd.notna(b)) & (b != 0), a / b, np.nan)


def calc_cv(x):
    x = pd.Series(x).dropna()
    m = x.mean()

    if len(x) < 2 or pd.isna(m) or m == 0:
        return np.nan

    return x.std() / m


# ============================================================
# LOAD AND CLEAN
# ============================================================

def load_and_clean(df):
    df = df.copy()

    df.columns = [str(c).strip() for c in df.columns]

    numeric_candidates = [
        TARGET,
        "normalized_cost",
        "normalized_cost_geography",
        "normalized_cost_shipment",
        "normalized_cost_density",
        "GEOGRAPHY_MULTIPLIER",
        "shipment_norm_multiplier_qt",
        "miles_per_stop_norm_multiplier_qt",
        "geo_mean",
        "STOP_COUNT_VAL_ROUTE_LVL",
        "TOTE_COUNT_VAL_ROUTE_LVL",
        "STOP_COUNT_VAL",
        "TOTE_COUNT_VAL",
        "ROUTE_COUNT_VAL",
        "DISTANCE_VAL",
        "LASTMILE_TOTAL_COST",
        "LASTMILE_BASE_COST",
        "LASTMILE_FUEL_COST",
        "LASTMILE_MISC_COST",
        "Cleansheet Cost Per Stop Conservative",
        "Cleansheet Cost Per Stop Aggressive",
        "year",
        "month",
    ]

    for c in existing_cols(df, numeric_candidates):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    text_candidates = [
        "DC_CD",
        "CARRIER_SCAC_CD",
        "DELIVERY_TYPE",
        "ASN_TYPE_CD",
        "ASN_TYPE_DESC",
        "Distribution Center",
    ]

    for c in existing_cols(df, text_candidates):
        df[c] = df[c].astype("string").fillna("Unknown")

    if TARGET not in df.columns:
        raise ValueError(f"Missing target column: {TARGET}")

    df["paid_flag"] = df[TARGET].fillna(0) > 0
    df["zero_cost_flag"] = df[TARGET].fillna(0) <= 0

    if "normalized_cost" in df.columns:
        df["normalization_ratio"] = safe_div(df["normalized_cost"], df[TARGET])
        df["normalization_sensitivity_pct"] = df["normalization_ratio"] - 1
    else:
        df["normalization_ratio"] = np.nan
        df["normalization_sensitivity_pct"] = np.nan

    if "Cleansheet Cost Per Stop Conservative" in df.columns:
        df["gap_vs_cleansheet_cons_pct"] = safe_div(
            df[TARGET],
            df["Cleansheet Cost Per Stop Conservative"]
        ) - 1
    else:
        df["gap_vs_cleansheet_cons_pct"] = np.nan

    if "Cleansheet Cost Per Stop Aggressive" in df.columns:
        df["gap_vs_cleansheet_aggr_pct"] = safe_div(
            df[TARGET],
            df["Cleansheet Cost Per Stop Aggressive"]
        ) - 1
    else:
        df["gap_vs_cleansheet_aggr_pct"] = np.nan

    return df

    

# ============================================================
# NORMALIZATION RELIABILITY
# ============================================================

def normalization_diagnostics(df, group_cols):
    metric_cols = existing_cols(
        df,
        [
            TARGET,
            "normalized_cost",
            "normalized_cost_geography",
            "normalized_cost_shipment",
            "normalized_cost_density",
        ],
    )

    cv_parts = []

    for metric in metric_cols:
        tmp = (
            df.groupby(group_cols, dropna=False)[metric]
            .agg(
                n="size",
                mean="mean",
                median="median",
                std="std",
                cv=calc_cv,
            )
            .reset_index()
        )

        tmp["metric"] = metric
        cv_parts.append(tmp)

    cv_table = pd.concat(cv_parts, ignore_index=True) if cv_parts else pd.DataFrame()

    rank_rows = []

    if {"DC_CD", "year", "month"}.issubset(df.columns):
        for metric in metric_cols:
            tmp = (
                df.groupby(["year", "month", "DC_CD"], dropna=False)[metric]
                .median()
                .reset_index()
            )

            tmp["period"] = pd.to_datetime(
                dict(
                    year=tmp["year"].astype(int),
                    month=tmp["month"].astype(int),
                    day=1,
                ),
                errors="coerce",
            )

            pivot = tmp.pivot_table(
                index="DC_CD",
                columns="period",
                values=metric,
                aggfunc="median",
            )

            periods = sorted([p for p in pivot.columns if pd.notna(p)])
            corr_values = []

            for i in range(1, len(periods)):
                a = pivot[periods[i - 1]]
                b = pivot[periods[i]]

                ok = a.notna() & b.notna()

                if ok.sum() >= 5:
                    corr_values.append(
                        a[ok].rank().corr(
                            b[ok].rank(),
                            method="spearman",
                        )
                    )

            rank_rows.append(
                {
                    "metric": metric,
                    "avg_month_to_month_rank_corr": np.nanmean(corr_values)
                    if corr_values
                    else np.nan,
                    "period_pairs_used": len(corr_values),
                }
            )

    rank_table = pd.DataFrame(rank_rows)

    return cv_table, rank_table

    # ============================================================
# MODEL FEATURES
# ============================================================

def get_model_columns(df):
    numeric_features = existing_cols(
        df,
        [
            "STOP_COUNT_VAL_ROUTE_LVL",
            "TOTE_COUNT_VAL_ROUTE_LVL",
            "STOP_COUNT_VAL",
            "TOTE_COUNT_VAL",
            "ROUTE_COUNT_VAL",
            "DISTANCE_VAL",
            "GEOGRAPHY_MULTIPLIER",
            "shipment_norm_multiplier_qt",
            "miles_per_stop_norm_multiplier_qt",
            "geo_mean",
            "year",
            "month",
        ],
    )

    categorical_features = existing_cols(
        df,
        [
            "DC_CD",
            "CARRIER_SCAC_CD",
            "DELIVERY_TYPE",
            "ASN_TYPE_CD",
            "ASN_TYPE_DESC",
        ],
    )

    return numeric_features, categorical_features


def make_preprocessor(numeric_features, categorical_features):
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
    )

    return preprocessor


def make_quantile_model(q, numeric_features, categorical_features):
    model = Pipeline(
        steps=[
            (
                "prep",
                make_preprocessor(
                    numeric_features,
                    categorical_features,
                ),
            ),
            (
                "model",
                HistGradientBoostingRegressor(
                    loss="quantile",
                    quantile=q,
                    max_iter=240,
                    learning_rate=0.06,
                    max_leaf_nodes=31,
                    l2_regularization=0.05,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    return model


# ============================================================
# FIT EXPECTED COST MODELS
# ============================================================

def fit_expected_cost_models(df):
    paid = df[
        df["paid_flag"]
        & df[TARGET].notna()
        & (df[TARGET] > 0)
    ].copy()

    if len(paid) < 200:
        raise ValueError("Not enough non-zero paid rows to train model.")

    if MAX_MODEL_ROWS is not None and len(paid) > MAX_MODEL_ROWS:
        model_df = paid.sample(
            MAX_MODEL_ROWS,
            random_state=RANDOM_STATE,
        )
    else:
        model_df = paid.copy()

    numeric_features, categorical_features = get_model_columns(model_df)
    feature_cols = numeric_features + categorical_features

    if len(feature_cols) == 0:
        raise ValueError("No usable model features found.")

    X = model_df[feature_cols]
    y = np.log1p(model_df[TARGET].clip(lower=0))

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]

    models = {}

    for q in quantiles:
        m = make_quantile_model(
            q,
            numeric_features,
            categorical_features,
        )

        m.fit(X_train, y_train)
        models[q] = m

    pred_p50_log = models[0.50].predict(X_test)

    y_test_cost = np.expm1(y_test)
    pred_p50_cost = np.expm1(pred_p50_log).clip(min=0)

    model_metrics = {
        "paid_rows_total": int(len(paid)),
        "model_rows_used": int(len(model_df)),
        "feature_count": int(len(feature_cols)),
        "mae_cost": float(mean_absolute_error(y_test_cost, pred_p50_cost)),
        "median_abs_error_cost": float(median_absolute_error(y_test_cost, pred_p50_cost)),
        "r2_log_target": float(r2_score(y_test, pred_p50_log)),
    }

    return models, feature_cols, model_metrics


# ============================================================
# SCORE ROWS
# ============================================================

def score_rows(df, models, feature_cols):
    scored = df.copy()

    eligible = (
        scored["paid_flag"]
        & scored[feature_cols].notna().any(axis=1)
    )

    for q, model in models.items():
        col = f"pred_p{int(q * 100):02d}"
        scored[col] = np.nan

        if eligible.sum() > 0:
            scored.loc[eligible, col] = np.expm1(
                model.predict(scored.loc[eligible, feature_cols])
            ).clip(min=0)

    qcols = [
        "pred_p10",
        "pred_p25",
        "pred_p50",
        "pred_p75",
        "pred_p90",
    ]

    if set(qcols).issubset(scored.columns):
        scored[qcols] = np.sort(scored[qcols].to_numpy(), axis=1)

    scored["row_residual"] = scored[TARGET] - scored["pred_p50"]
    scored["row_residual_pct"] = safe_div(scored[TARGET], scored["pred_p50"]) - 1

    return scored


# ============================================================
# MARKET SCORECARD
# ============================================================

def build_scorecard(scored, group_cols):
    group_cols = existing_cols(scored, group_cols)

    if len(group_cols) == 0:
        group_cols = ["DC_CD"] if "DC_CD" in scored.columns else []

    if len(group_cols) == 0:
        raise ValueError("No grouping columns available.")

    aggs = {
        "records": (TARGET, "size"),
        "paid_records": ("paid_flag", "sum"),
        "zero_records": ("zero_cost_flag", "sum"),

        "actual_mean_cps": (TARGET, "mean"),
        "actual_median_cps": (TARGET, "median"),
        "actual_p75_cps": (TARGET, lambda x: x.quantile(0.75)),
        "actual_p90_cps": (TARGET, lambda x: x.quantile(0.90)),

        "pred_p10": ("pred_p10", "median"),
        "pred_p25": ("pred_p25", "median"),
        "pred_p50": ("pred_p50", "median"),
        "pred_p75": ("pred_p75", "median"),
        "pred_p90": ("pred_p90", "median"),

        "median_row_residual_pct": ("row_residual_pct", "median"),

        "norm_sensitivity_median_pct": (
            "normalization_sensitivity_pct",
            "median",
        ),

        "norm_sensitivity_abs_median_pct": (
            "normalization_sensitivity_pct",
            lambda x: np.nanmedian(np.abs(x)),
        ),

        "cleansheet_cons_gap_median_pct": (
            "gap_vs_cleansheet_cons_pct",
            "median",
        ),

        "cleansheet_aggr_gap_median_pct": (
            "gap_vs_cleansheet_aggr_pct",
            "median",
        ),
    }

    optional_medians = [
        "normalized_cost",
        "normalized_cost_geography",
        "normalized_cost_shipment",
        "normalized_cost_density",
        "GEOGRAPHY_MULTIPLIER",
        "shipment_norm_multiplier_qt",
        "miles_per_stop_norm_multiplier_qt",
        "DISTANCE_VAL",
        "STOP_COUNT_VAL_ROUTE_LVL",
        "TOTE_COUNT_VAL_ROUTE_LVL",
        "Cleansheet Cost Per Stop Conservative",
        "Cleansheet Cost Per Stop Aggressive",
    ]

    for c in existing_cols(scored, optional_medians):
        clean_name = c.lower().replace(" ", "_").replace("/", "_")
        aggs[f"median_{clean_name}"] = (c, "median")

    scorecard = (
        scored
        .groupby(group_cols, dropna=False)
        .agg(**aggs)
        .reset_index()
    )

    scorecard["zero_rate"] = safe_div(
        scorecard["zero_records"],
        scorecard["records"],
    )

    scorecard["paid_rate"] = safe_div(
        scorecard["paid_records"],
        scorecard["records"],
    )

    scorecard["model_residual"] = (
        scorecard["actual_median_cps"]
        - scorecard["pred_p50"]
    )

    scorecard["model_residual_pct"] = safe_div(
        scorecard["actual_median_cps"],
        scorecard["pred_p50"],
    ) - 1

    scorecard["above_p90"] = scorecard["actual_median_cps"] > scorecard["pred_p90"]
    scorecard["above_p75"] = scorecard["actual_median_cps"] > scorecard["pred_p75"]
    scorecard["below_p10"] = scorecard["actual_median_cps"] < scorecard["pred_p10"]
    scorecard["below_p25"] = scorecard["actual_median_cps"] < scorecard["pred_p25"]

    scorecard["confidence_score"] = 0

    scorecard.loc[
        scorecard["paid_records"] >= MIN_GROUP_N,
        "confidence_score",
    ] += 1

    scorecard.loc[
        scorecard["paid_records"] >= MIN_GROUP_N * 3,
        "confidence_score",
    ] += 1

    scorecard.loc[
        scorecard["zero_rate"] <= 0.35,
        "confidence_score",
    ] += 1

    scorecard.loc[
        scorecard["norm_sensitivity_abs_median_pct"].fillna(0) <= 0.40,
        "confidence_score",
    ] += 1

    scorecard["confidence"] = np.select(
        [
            scorecard["confidence_score"] >= 4,
            scorecard["confidence_score"] == 3,
            scorecard["confidence_score"] == 2,
        ],
        [
            "High",
            "Medium",
            "Low",
        ],
        default="Very Low",
    )

    scorecard["classification"] = "Normal / inside expected band"

    strong_conf = scorecard["confidence"].isin(["High", "Medium"])

    scorecard.loc[
        (scorecard["model_residual_pct"] >= OVERPAY_STRONG)
        & scorecard["above_p90"]
        & strong_conf,
        "classification",
    ] = "Strong overpay candidate"

    scorecard.loc[
        scorecard["classification"].eq("Normal / inside expected band")
        & (scorecard["model_residual_pct"] >= OVERPAY_POSSIBLE)
        & scorecard["above_p75"],
        "classification",
    ] = "Possible overpay"

    scorecard.loc[
        (scorecard["model_residual_pct"] <= UNDERPAY_STRONG)
        & scorecard["below_p10"]
        & strong_conf,
        "classification",
    ] = "Strong underpay candidate"

    scorecard.loc[
        scorecard["classification"].eq("Normal / inside expected band")
        & (scorecard["model_residual_pct"] <= UNDERPAY_POSSIBLE)
        & scorecard["below_p25"],
        "classification",
    ] = "Possible underpay"

    scorecard.loc[
        (scorecard["paid_records"] < MIN_GROUP_N)
        | scorecard["confidence"].eq("Very Low"),
        "classification",
    ] = "Not enough evidence"

    scorecard["cleansheet_overpay_support"] = (
        scorecard["cleansheet_aggr_gap_median_pct"].notna()
        & (scorecard["cleansheet_aggr_gap_median_pct"] > 0)
    )

    scorecard["cleansheet_underpay_support"] = (
        scorecard["cleansheet_cons_gap_median_pct"].notna()
        & (scorecard["cleansheet_cons_gap_median_pct"] < 0)
    )

    scorecard["business_note"] = np.select(
        [
            scorecard["classification"].str.contains(
                "overpay",
                case=False,
                na=False,
            )
            & scorecard["cleansheet_overpay_support"],

            scorecard["classification"].str.contains(
                "underpay",
                case=False,
                na=False,
            )
            & scorecard["cleansheet_underpay_support"],

            scorecard["classification"].eq("Not enough evidence"),

            scorecard["norm_sensitivity_abs_median_pct"].fillna(0) > 0.40,
        ],
        [
            "Model and cleansheet both point high",
            "Model and cleansheet both point low",
            "Low volume, high zero rate, or weak basis",
            "Sensitive to multiplier assumptions",
        ],
        default="Model-based residual signal",
    )

    return scorecard


# ============================================================
# ANOMALY OVERLAY
# ============================================================

def add_anomaly_score(scorecard):
    features = existing_cols(
        scorecard,
        [
            "actual_median_cps",
            "pred_p50",
            "model_residual_pct",
            "zero_rate",
            "norm_sensitivity_abs_median_pct",
            "paid_records",
            "median_distance_val",
            "median_stop_count_val_route_lvl",
            "median_tote_count_val_route_lvl",
        ],
    )

    if len(scorecard) < 20 or len(features) < 3:
        scorecard["anomaly_score"] = np.nan
        scorecard["anomaly_flag"] = False
        return scorecard

    X = (
        scorecard[features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(scorecard[features].median(numeric_only=True))
    )

    iso = IsolationForest(
        n_estimators=200,
        contamination="auto",
        random_state=RANDOM_STATE,
    )

    scorecard["anomaly_score"] = -iso.fit(X).score_samples(X)
    scorecard["anomaly_flag"] = (
        scorecard["anomaly_score"]
        >= scorecard["anomaly_score"].quantile(0.90)
    )

    return scorecard

In [0]:
# ============================================================
# FULL PIPELINE
# ============================================================

df = load_and_clean(LM_CS_df)

GROUP_COLS = existing_cols(df, GROUP_COLS)

if len(GROUP_COLS) == 0:
    GROUP_COLS = ["DC_CD"] if "DC_CD" in df.columns else []

print("Rows:", f"{len(df):,}")
print("Columns:", f"{len(df.columns):,}")
print("Grouping:", GROUP_COLS)
print("Paid rows:", f"{df['paid_flag'].sum():,}")
print("Zero cost rows:", f"{df['zero_cost_flag'].sum():,}")
print("Zero cost rate:", f"{df['zero_cost_flag'].mean():.1%}")

cv_table, rank_table = normalization_diagnostics(df, GROUP_COLS)

models, feature_cols, model_metrics = fit_expected_cost_models(df)

scored = score_rows(df, models, feature_cols)

scorecard = build_scorecard(scored, GROUP_COLS)

scorecard = add_anomaly_score(scorecard)

scorecard = scorecard.sort_values(
    [
        "classification",
        "model_residual_pct",
        "paid_records",
    ],
    ascending=[
        True,
        False,
        False,
    ],
)

if EXPORT_EXCEL:
    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="openpyxl") as writer:
        scorecard.to_excel(writer, sheet_name="scorecard", index=False)
        cv_table.to_excel(writer, sheet_name="normalization_cv", index=False)
        rank_table.to_excel(writer, sheet_name="rank_stability", index=False)
        pd.DataFrame([model_metrics]).to_excel(
            writer,
            sheet_name="model_metrics",
            index=False,
        )

    print("Excel exported:", EXCEL_OUTPUT_PATH)

print("\nModel metrics:")
display(pd.DataFrame([model_metrics]))

print("\nClassification counts:")
display(
    scorecard["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="count")
)

print("\nTop scorecard rows:")
display(scorecard.head(25))

In [0]:



# ============================================================
# VISUAL 1: CLASSIFICATION COUNTS
# ============================================================

classification_counts = (
    scorecard["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="count")
)

fig = px.bar(
    classification_counts,
    x="classification",
    y="count",
    color="classification",
    title="Outlier Classification Counts",
    text="count",
)

fig.update_layout(
    xaxis_title="Classification",
    yaxis_title="Number of Market Groups",
    showlegend=False,
)

fig.show()


# ============================================================
# VISUAL 2: RESIDUAL VS PAID RECORDS
# ============================================================

hover_cols = existing_cols(
    scorecard,
    [
        "DC_CD",
        "Distribution Center",
        "year",
        "month",
        "CARRIER_SCAC_CD",
        "actual_median_cps",
        "pred_p50",
        "pred_p90",
        "model_residual_pct",
        "confidence",
        "business_note",
    ],
)

fig = px.scatter(
    scorecard,
    x="paid_records",
    y="model_residual_pct",
    color="classification",
    size="actual_median_cps",
    hover_data=hover_cols,
    title="Model Residual Percent vs Paid Records",
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)

fig.add_hline(
    y=OVERPAY_POSSIBLE,
    line_dash="dot",
    line_color="red",
)

fig.add_hline(
    y=UNDERPAY_POSSIBLE,
    line_dash="dot",
    line_color="blue",
)

fig.update_layout(
    xaxis_title="Paid Records",
    yaxis_title="Actual Median CPS / Expected Median CPS - 1",
)

fig.show()


# ============================================================
# VISUAL 3: TOP OVERPAY CANDIDATES
# ============================================================

top_overpay = (
    scorecard[
        scorecard["classification"].isin(
            [
                "Strong overpay candidate",
                "Possible overpay",
            ]
        )
    ]
    .sort_values("model_residual_pct", ascending=False)
    .head(25)
    .copy()
)

if len(top_overpay) > 0:
    label_col = "DC_CD" if "DC_CD" in top_overpay.columns else GROUP_COLS[0]

    top_overpay["label"] = top_overpay[label_col].astype(str)

    if "CARRIER_SCAC_CD" in top_overpay.columns:
        top_overpay["label"] = (
            top_overpay["label"]
            + " | "
            + top_overpay["CARRIER_SCAC_CD"].astype(str)
        )

    if "month" in top_overpay.columns:
        top_overpay["label"] = (
            top_overpay["label"]
            + " | M"
            + top_overpay["month"].astype(str)
        )

    fig = px.bar(
        top_overpay.sort_values("model_residual_pct"),
        x="model_residual_pct",
        y="label",
        color="classification",
        orientation="h",
        hover_data=hover_cols,
        title="Top Overpay Candidates by Model Residual Percent",
    )

    fig.update_layout(
        xaxis_title="Residual Percent",
        yaxis_title="Market Group",
    )

    fig.show()


# ============================================================
# VISUAL 4: TOP UNDERPAY CANDIDATES
# ============================================================

top_underpay = (
    scorecard[
        scorecard["classification"].isin(
            [
                "Strong underpay candidate",
                "Possible underpay",
            ]
        )
    ]
    .sort_values("model_residual_pct", ascending=True)
    .head(25)
    .copy()
)

if len(top_underpay) > 0:
    label_col = "DC_CD" if "DC_CD" in top_underpay.columns else GROUP_COLS[0]

    top_underpay["label"] = top_underpay[label_col].astype(str)

    if "CARRIER_SCAC_CD" in top_underpay.columns:
        top_underpay["label"] = (
            top_underpay["label"]
            + " | "
            + top_underpay["CARRIER_SCAC_CD"].astype(str)
        )

    if "month" in top_underpay.columns:
        top_underpay["label"] = (
            top_underpay["label"]
            + " | M"
            + top_underpay["month"].astype(str)
        )

    fig = px.bar(
        top_underpay.sort_values("model_residual_pct", ascending=False),
        x="model_residual_pct",
        y="label",
        color="classification",
        orientation="h",
        hover_data=hover_cols,
        title="Top Underpay Candidates by Model Residual Percent",
    )

    fig.update_layout(
        xaxis_title="Residual Percent",
        yaxis_title="Market Group",
    )

    fig.show()


# ============================================================
# VISUAL 5: EXPECTED COST BAND VS ACTUAL
# ============================================================

band_sample = (
    scorecard
    .sort_values("model_residual_pct", ascending=False)
    .head(40)
    .copy()
)

if len(band_sample) > 0:
    label_col = "DC_CD" if "DC_CD" in band_sample.columns else GROUP_COLS[0]

    band_sample["label"] = band_sample[label_col].astype(str)

    if "CARRIER_SCAC_CD" in band_sample.columns:
        band_sample["label"] = (
            band_sample["label"]
            + " | "
            + band_sample["CARRIER_SCAC_CD"].astype(str)
        )

    if "month" in band_sample.columns:
        band_sample["label"] = (
            band_sample["label"]
            + " | M"
            + band_sample["month"].astype(str)
        )

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=band_sample["label"],
            y=band_sample["actual_median_cps"],
            name="Actual median CPS",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=band_sample["label"],
            y=band_sample["pred_p50"],
            name="Expected P50",
            mode="lines+markers",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=band_sample["label"],
            y=band_sample["pred_p90"],
            name="Expected P90",
            mode="lines+markers",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=band_sample["label"],
            y=band_sample["pred_p10"],
            name="Expected P10",
            mode="lines+markers",
        )
    )

    fig.update_layout(
        title="Actual CPS vs Expected Cost Band, Top Positive Residual Groups",
        xaxis_title="Market Group",
        yaxis_title="Cost Per Stop",
        xaxis_tickangle=-45,
        barmode="group",
        height=650,
    )

    fig.show()


# ============================================================
# VISUAL 6: HEATMAP BY DC AND MONTH
# ============================================================

if {"DC_CD", "month", "model_residual_pct"}.issubset(scorecard.columns):
    heat = scorecard.pivot_table(
        index="DC_CD",
        columns="month",
        values="model_residual_pct",
        aggfunc="median",
    )

    fig = px.imshow(
        heat,
        aspect="auto",
        color_continuous_scale="RdBu_r",
        title="Median Model Residual Percent by DC and Month",
    )

    fig.update_layout(
        xaxis_title="Month",
        yaxis_title="DC",
    )

    fig.show()


# ============================================================
# VISUAL 7: NORMALIZATION CV COMPARISON
# ============================================================

if not cv_table.empty:
    cv_summary = (
        cv_table
        .groupby("metric", as_index=False)["cv"]
        .median()
        .sort_values("cv")
    )

    fig = px.bar(
        cv_summary,
        x="metric",
        y="cv",
        color="metric",
        title="Median Coefficient of Variation by Cost Metric",
        text_auto=".2f",
    )

    fig.update_layout(
        xaxis_title="Cost Metric",
        yaxis_title="Median CV",
        showlegend=False,
    )

    fig.show()


# ============================================================
# VISUAL 8: RANK STABILITY
# ============================================================

if not rank_table.empty:
    fig = px.bar(
        rank_table.sort_values("avg_month_to_month_rank_corr", ascending=False),
        x="metric",
        y="avg_month_to_month_rank_corr",
        color="metric",
        title="Month-to-Month Rank Stability by Cost Metric",
        text_auto=".2f",
    )

    fig.update_layout(
        xaxis_title="Cost Metric",
        yaxis_title="Average Spearman Rank Correlation",
        showlegend=False,
    )

    fig.show()


# ============================================================
# VISUAL 9: MULTIPLIER SENSITIVITY VS RESIDUAL
# ============================================================

if "norm_sensitivity_abs_median_pct" in scorecard.columns:
    fig = px.scatter(
        scorecard,
        x="norm_sensitivity_abs_median_pct",
        y="model_residual_pct",
        color="classification",
        size="paid_records",
        hover_data=hover_cols,
        title="Multiplier Sensitivity vs Model Residual Percent",
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black",
    )

    fig.add_vline(
        x=0.40,
        line_dash="dot",
        line_color="orange",
    )

    fig.update_layout(
        xaxis_title="Median Absolute Normalization Sensitivity",
        yaxis_title="Model Residual Percent",
    )

    fig.show()


# ============================================================
# VISUAL 10: CLEANSHEET TRIANGULATION
# ============================================================

if "cleansheet_aggr_gap_median_pct" in scorecard.columns:
    cleansheet_view = scorecard[
        scorecard["cleansheet_aggr_gap_median_pct"].notna()
    ].copy()

    if len(cleansheet_view) > 0:
        fig = px.scatter(
            cleansheet_view,
            x="cleansheet_aggr_gap_median_pct",
            y="model_residual_pct",
            color="classification",
            size="paid_records",
            hover_data=hover_cols,
            title="Model Residual vs Cleansheet Aggressive Gap",
        )

        fig.add_hline(
            y=0,
            line_dash="dash",
            line_color="black",
        )

        fig.add_vline(
            x=0,
            line_dash="dash",
            line_color="black",
        )

        fig.update_layout(
            xaxis_title="Actual / Cleansheet Aggressive - 1",
            yaxis_title="Actual / Model Expected P50 - 1",
        )

        fig.show()


# ============================================================
# VISUAL 11: ANOMALY SCORE
# ============================================================

if "anomaly_score" in scorecard.columns:
    fig = px.scatter(
        scorecard,
        x="model_residual_pct",
        y="anomaly_score",
        color="classification",
        size="paid_records",
        hover_data=hover_cols,
        title="Residual Percent vs Anomaly Score",
    )

    fig.add_vline(
        x=0,
        line_dash="dash",
        line_color="black",
    )

    fig.update_layout(
        xaxis_title="Model Residual Percent",
        yaxis_title="Anomaly Score",
    )

    fig.show()


# ============================================================
# USEFUL FINAL TABLES
# ============================================================

display_cols = existing_cols(
    scorecard,
    [
        "DC_CD",
        "Distribution Center",
        "year",
        "month",
        "CARRIER_SCAC_CD",
        "classification",
        "confidence",
        "business_note",
        "records",
        "paid_records",
        "zero_rate",
        "actual_median_cps",
        "pred_p10",
        "pred_p25",
        "pred_p50",
        "pred_p75",
        "pred_p90",
        "model_residual",
        "model_residual_pct",
        "norm_sensitivity_abs_median_pct",
        "cleansheet_cons_gap_median_pct",
        "cleansheet_aggr_gap_median_pct",
        "anomaly_score",
        "anomaly_flag",
    ],
)

print("\nStrong overpay candidates")
display(
    scorecard[
        scorecard["classification"].eq("Strong overpay candidate")
    ][display_cols]
    .sort_values("model_residual_pct", ascending=False)
    .head(50)
)

print("\nPossible overpay candidates")
display(
    scorecard[
        scorecard["classification"].eq("Possible overpay")
    ][display_cols]
    .sort_values("model_residual_pct", ascending=False)
    .head(50)
)

print("\nUnderpay candidates")
display(
    scorecard[
        scorecard["classification"].str.contains("underpay", case=False, na=False)
    ][display_cols]
    .sort_values("model_residual_pct", ascending=True)
    .head(50)
)

print("\nNot enough evidence")
display(
    scorecard[
        scorecard["classification"].eq("Not enough evidence")
    ][display_cols]
    .sort_values("paid_records", ascending=False)
    .head(50)
)